In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import time

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cpu


In [2]:
cache = torch.load("embeddings.pth", map_location="cpu")

X_train = cache["train"]["embeddings"].float()
y_train = cache["train"]["labels"]

X_val   = cache["val"]["embeddings"].float()
y_val   = cache["val"]["labels"]

X_test  = cache["test"]["embeddings"].float()
y_test  = cache["test"]["labels"]


In [3]:
y_train = y_train.squeeze().long()
y_val   = y_val.squeeze().long()
y_test  = y_test.squeeze().long()


In [4]:
print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)

assert X_train.ndim == 2
assert y_train.ndim == 1
assert len(X_train) == len(y_train)


Train: torch.Size([89996, 512]) torch.Size([89996])
Val:   torch.Size([10004, 512]) torch.Size([10004])
Test:  torch.Size([7180, 512]) torch.Size([7180])


In [5]:
class EmbeddingDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [6]:
batch_size = 128

train_loader = DataLoader(
    EmbeddingDataset(X_train, y_train),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    EmbeddingDataset(X_val, y_val),
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    EmbeddingDataset(X_test, y_test),
    batch_size=batch_size,
    shuffle=False
)


In [7]:
embed_dim = X_train.shape[1]      # should be 512
num_classes = len(torch.unique(y_train))

linear_probe = nn.Linear(embed_dim, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(linear_probe.parameters(), lr=1e-3)

print(linear_probe)


Linear(in_features=512, out_features=9, bias=True)


In [9]:
def run_epoch(model, loader, train=False):
    model.train() if train else model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.set_grad_enabled(train):
        for X, y in loader:
            X = X.to(device)
            y = y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * y.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss / total, correct / total


In [18]:
epochs = 40
history = {"train_acc": [], "val_acc": []}

start = time.time()

for epoch in range(1, epochs + 1):
    tr_loss, tr_acc = run_epoch(linear_probe, train_loader, train=True)
    va_loss, va_acc = run_epoch(linear_probe, val_loader)

    history["train_acc"].append(tr_acc)
    history["val_acc"].append(va_acc)

    print(
        f"Epoch [{epoch:02d}/{epochs}] | "
        f"Train Acc: {tr_acc*100:.2f}% | "
        f"Val Acc: {va_acc*100:.2f}%"
    )

print(f"⏱ Training time: {(time.time()-start)/60:.2f} min")


Epoch [01/40] | Train Acc: 90.44% | Val Acc: 90.62%
Epoch [02/40] | Train Acc: 90.49% | Val Acc: 90.56%
Epoch [03/40] | Train Acc: 90.51% | Val Acc: 90.78%
Epoch [04/40] | Train Acc: 90.52% | Val Acc: 90.71%
Epoch [05/40] | Train Acc: 90.52% | Val Acc: 90.63%
Epoch [06/40] | Train Acc: 90.59% | Val Acc: 90.72%
Epoch [07/40] | Train Acc: 90.59% | Val Acc: 90.82%
Epoch [08/40] | Train Acc: 90.65% | Val Acc: 90.77%
Epoch [09/40] | Train Acc: 90.63% | Val Acc: 90.83%
Epoch [10/40] | Train Acc: 90.67% | Val Acc: 90.86%
Epoch [11/40] | Train Acc: 90.71% | Val Acc: 90.89%
Epoch [12/40] | Train Acc: 90.73% | Val Acc: 90.93%
Epoch [13/40] | Train Acc: 90.77% | Val Acc: 90.99%
Epoch [14/40] | Train Acc: 90.76% | Val Acc: 90.86%
Epoch [15/40] | Train Acc: 90.82% | Val Acc: 90.99%
Epoch [16/40] | Train Acc: 90.82% | Val Acc: 90.95%
Epoch [17/40] | Train Acc: 90.83% | Val Acc: 91.01%
Epoch [18/40] | Train Acc: 90.84% | Val Acc: 91.08%
Epoch [19/40] | Train Acc: 90.87% | Val Acc: 90.96%
Epoch [20/40

In [19]:
linear_probe.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        preds = linear_probe(X).argmax(dim=1).cpu()
        all_preds.append(preds)
        all_labels.append(y)

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()


In [20]:
test_acc = accuracy_score(all_labels, all_preds)
macro_f1 = f1_score(all_labels, all_preds, average="macro")
cm = confusion_matrix(all_labels, all_preds)

print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"Macro F1:      {macro_f1:.4f}")
print("Confusion Matrix:\n", cm)


Test Accuracy: 87.91%
Macro F1:      0.8316
Confusion Matrix:
 [[1314    1    1    0    6   12    3    0    1]
 [   6  841    0    0    0    0    0    0    0]
 [   0    1  203    0    0  114    0   18    3]
 [   0    0    1  597    0    1   10    1   24]
 [  11   41    1    0  960    2    8    6    6]
 [   0    5   17    0   22  434    1  106    7]
 [   1    0    1   16    5    1  639    0   78]
 [   0    1   41    2    9   90    7  220   51]
 [   0    0   23   37   12    4   48    5 1104]]
